# IDS Challenge: Teilprojekt 01 Optimierung
## Ergebnisse 

### Gruppe: ___
### Tutor: ___

Das ist Ihr Arbeitsbereich - Implementieren Sie hier alle Abnahme-relevanten Funktionen und Elemente und fügen Sie kurze textuelle Beschreibungen hinzu, die Ihre Schritte erklären. (Siehe `README.ipynb`)

Imports & Konstanten

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

TIME_LIMIT = 5 * 3600

def calc_total_tour_time(sequ, matrix):
    total_time = 0
    for i in range(len(sequ) - 1):
        total_time += matrix[sequ[i], sequ[i + 1]]
    return total_time

Datenbasisfunktionen

In [ ]:
def read_ls_matrix(pfad="data/ls_matrix_55_42.txt"):
    matrix = {}
    with open(pfad) as f:
        for zeile in f:
            teile = zeile.strip().split(";")
            von = int(teile[0])
            nach = int(teile[1])
            zeit = float(teile[2])
            matrix[(von, nach)] = zeit
    return matrix

def read_points(pfad):
    points = {}
    with open(pfad) as f:
        for zeile in f:
            teile = zeile.strip().split(";")
            index = int(teile[0])
            x = float(teile[1])
            y = float(teile[2])
            stockwerk = int(teile[3])
            points[index] = (x, y, stockwerk)
    return points

def transform_data():
    matrix = read_ls_matrix("data/ls_matrix_55_42.txt")

    points_gianni = read_points("data/points_gianni_55_42.txt")
    points_lissi = read_points("data/points_lissi_55_42.txt")
    points_ben = read_points("data/points_ben_55_42.txt")

    points = {}
    points.update(points_gianni)
    points.update(points_lissi)
    points.update(points_ben)

    return points, matrix

Clustering mit K-Means in 3 Gruppen

In [ ]:
from sklearn.cluster import KMeans

def cluster_machines(points, k=3):
    maschinen_indizes = [index for index in points if index != 0]
    koordinaten = np.array([points[i][:2] for i in maschinen_indizes])
    kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
    labels = kmeans.fit_predict(koordinaten)
    clusters = [[] for _ in range(k)]
    for maschine, label in zip(maschinen_indizes, labels):
        clusters[label].append(maschine)
    return clusters

Nutzwert & Auswahl

In [ ]:
def depot_distance(machine, matrix):
    return matrix[(0, machine)]

def instertion_cost(machine, tour, matrix):
    location = tour[-1]
    return matrix[(location, machine)]

def machine_value(machine, tour, matrix, w):
    fern = depot_distance(machine, matrix)
    guenstig = instertion_cost(machine, tour, matrix)
    return w * fern - (1 - w) * guenstig

def best_value_unvisited(noch_offen, tour, matrix, w):
    best_machine = None
    best_value = None
    for machine in noch_offen:
        value = machine_value(machine, tour, matrix, w)
        if best_value is None or value > best_value:
            best_value = value
            best_machine = machine
    return best_machine

Tour

In [ ]:
def fits_in_time(tour, maschine, matrix):
    test_tour = tour + [maschine, 0]
    return calc_total_tour_time(test_tour, matrix) <= TIME_LIMIT

def build_tour(cluster, matrix):
    tour = [0]
    noch_offen = cluster.copy()
    while noch_offen:
        naechste = best_value_unvisited(noch_offen, tour, matrix, w)
        if fits_in_time(tour, naechste, matrix):
            tour.append(naechste)
            noch_offen.remove(naechste)
        else:
            break
    tour.append(0)
    return tour

def build_all_tours(clusters, matrix, w):
    tours = []
    for cluster in clusters:
        tour = build_tour(cluster, matrix, w)
        tours.append(tour)
    return tours
    

Bewertung & Trade-off

In [ ]:
def remaining_machines(tours, alle_maschinen):
    besucht = set()
    for tour in tours:
        for punkt in tour:
            besucht.add(punkt)
    return [m for m in alle_maschinen if m not in besucht]


def sum_individual_trips(remaining, matrix):
    summe = 0
    for maschine in remaining:
        summe += matrix[(0, maschine)]
        summe += matrix[(maschine, 0)]
    return summe


def evaluate_solution(tours, alle_maschinen, matrix):
    remaining = remaining_machines(tours, alle_maschinen)
    abdeckung = len(alle_maschinen) - len(remaining)
    rest_fussweg = sum_individual_trips(remaining, matrix)
    return (abdeckung, rest_fussweg)

Orchestrierung

In [ ]:

def tune_weights(clusters, alle_maschinen, matrix):
    best_score = None
    best_w = None
    best_tours = None
    for w in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        tours = build_all_tours(clusters, matrix, w)
        abdeckung, rest_fussweg = evaluate_solution(tours, alle_maschinen, matrix)
        score = (abdeckung, -rest_fussweg)
        if best_score is None or score > best_score:
            best_score = score
            best_w = w
            best_tours = tours
    return best_w, best_tours

Ergebnis

In [ ]:
def remaining_machines(tours, alle_maschinen):
    besucht = set()
    for tour in tours:
        for punkt in tour:
            besucht.add(punkt)
    return [m for m in alle_maschinen if m not in besucht]


def sum_individual_trips(remaining, matrix):
    summe = 0
    for maschine in remaining:
        summe += matrix[(0, maschine)]
        summe += matrix[(maschine, 0)]
    return summe


def evaluate_solution(tours, alle_maschinen, matrix):
    remaining = remaining_machines(tours, alle_maschinen)
    abdeckung = len(alle_maschinen) - len(remaining)
    rest_fussweg = sum_individual_trips(remaining, matrix)
    return (abdeckung, rest_fussweg)


def print_solution(tours, abdeckung, remaining, rest_fussweg):
    print(f"Abgedeckte Maschinen: {abdeckung}")
    for i, tour in enumerate(tours):
        print(f"Roboter {i + 1}: {tour}")
    print(f"Übrige Maschinen ({len(remaining)}): {remaining}")
    print(f"Restfußweg: {rest_fussweg} s  ({rest_fussweg / 3600:.2f} h)")


def report_results(tours, alle_maschinen, matrix):
    abdeckung, rest_fussweg = evaluate_solution(tours, alle_maschinen, matrix)
    remaining = remaining_machines(tours, alle_maschinen)
    print_solution(tours, abdeckung, remaining, rest_fussweg)
    return tours, abdeckung, remaining, rest_fussweg

In [ ]:
def solve():
    points, matrix = transform_data()
    all_machines =[]
    for index in points:
        if index != 0:
            all_machines.append(index)
    clusters = cluster_machines(points)
    best_w, tours = tune_weights(clusters, all_machines, matrix)
    return report_results(tours, all_machines, matrix)
